# Tiny LLM Lab: MHA + KV cache + BPE

Notebook này chạy từng bước trên project `D:/tiny-llm-lab/llm-lab`. Mục tiêu là hiểu dữ liệu, MHA causal mask, một bước training, checkpoint/resume và KV-cache parity.

In [1]:
from pathlib import Path
import json, sys, torch
LAB_ROOT = Path(r'D:/tiny-llm-lab/llm-lab')
sys.path.insert(0, str(LAB_ROOT / 'src'))
from llm_lab.config import ModelConfig, TrainingConfig
from llm_lab.data import BPETokenizer, build_manifest, make_loaders, read_documents, split_documents
from llm_lab.model import GPTModel, MultiHeadAttention, count_parameters
from llm_lab.training import evaluate, loss_for_batch, load_checkpoint, save_checkpoint, train
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('project:', LAB_ROOT)
print('device:', device)

project: D:\tiny-llm-lab\llm-lab
device: cpu


## 1. Documents, deterministic split và BPE

BPE được fit trên train documents. Validation documents không được dùng để học merge rules.

In [2]:
data_path = LAB_ROOT / 'data' / 'tinystories_sample.jsonl'
documents = read_documents(data_path)
train_docs, val_docs = split_documents(documents, train_fraction=0.9, seed=42)
tokenizer = BPETokenizer.fit(train_docs, vocab_size=512, min_frequency=2)
train_ids = tokenizer.encode_documents(train_docs)
val_ids = tokenizer.encode_documents(val_docs)
source = {'id': 'roneneldan/TinyStories', 'url': 'https://huggingface.co/datasets/roneneldan/TinyStories', 'license': 'CDLA-Sharing-1.0', 'license_url': 'https://cdla.dev/sharing-1-0/'}
manifest = build_manifest(documents, train_docs, val_docs, source, seed=42, train_fraction=0.9)
print('documents:', len(documents), 'train:', len(train_docs), 'validation:', len(val_docs))
print('BPE vocab:', tokenizer.vocab_size, 'train tokens:', len(train_ids), 'validation tokens:', len(val_ids))
print('manifest:', manifest.to_dict())

documents: 1000 train: 900 validation: 100
BPE vocab: 512 train tokens: 367513 validation tokens: 42190
manifest: {'source_id': 'roneneldan/TinyStories', 'source_url': 'https://huggingface.co/datasets/roneneldan/TinyStories', 'license': 'CDLA-Sharing-1.0', 'license_url': 'https://cdla.dev/sharing-1-0/', 'document_count': 1000, 'document_sha256': 'fb247a3a673c88e0a9bfbf387dbd87466bde5bafd048a9e71ffd691512ba8f15', 'split_seed': 42, 'train_fraction': 0.9, 'train_document_count': 900, 'validation_document_count': 100}


## 2. Next-token batches

Mỗi input có shape `(batch, context)`. Target là cùng đoạn nhưng lệch phải một token.

In [3]:
cfg = ModelConfig(vocab_size=tokenizer.vocab_size, context_length=32, emb_dim=128, n_heads=4, n_layers=4, dropout=0.1)
train_loader, val_loader = make_loaders(train_ids, val_ids, cfg.context_length, batch_size=8, seed=42)
inputs, targets = next(iter(train_loader))
print('inputs:', tuple(inputs.shape), 'targets:', tuple(targets.shape), 'dtype:', inputs.dtype)
print('decoded input prefix:', tokenizer.decode(inputs[0, :16].tolist()))
print('decoded target prefix:', tokenizer.decode(targets[0, :16].tolist()))

inputs: (8, 32) targets: (8, 32) dtype: torch.int64
decoded input prefix:  need to learn to share. If you can't sh
decoded target prefix: ed to learn to share. If you can't share


## 3. MHA forward và causal mask

MHA tạo Q/K/V, chia thành heads, tính scaled dot-product attention, rồi mask mọi key ở tương lai.

In [4]:
torch.manual_seed(42)
attention = MultiHeadAttention(cfg).eval()
x = torch.randn(2, 6, cfg.emb_dim)
attention_output, no_cache = attention(x)
print('input:', tuple(x.shape), 'attention output:', tuple(attention_output.shape), 'cache:', no_cache)
changed_future = x.clone(); changed_future[:, 4:] = torch.randn_like(changed_future[:, 4:]) * 100
before, _ = attention(x); after, _ = attention(changed_future)
print('max prefix difference after changing future:', (before[:, :4] - after[:, :4]).abs().max().item())

input: (2, 6, 128) attention output: (2, 6, 128) cache: None
max prefix difference after changing future: 0.0


## 4. GPT baseline và một bước training

Training luôn gọi `use_cache=False`; KV cache chỉ dành cho inference.

In [5]:
model = GPTModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
batch_inputs, batch_targets = inputs.to(device), targets.to(device)
model.train(); loss_before = loss_for_batch(model, batch_inputs, batch_targets)
optimizer.zero_grad(set_to_none=True); loss_before.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
print('parameters:', count_parameters(model), 'loss before update:', loss_before.item())

parameters: 926976 loss before update: 6.400781631469727


In [ ]:
small_train_cfg = TrainingConfig(batch_size=8, max_steps=5, eval_every=5, eval_batches=2, save_every=5, seed=42)
history = train(model, train_loader, val_loader, small_train_cfg, device)
print('history:', history)
print('token-weighted validation loss:', evaluate(model, val_loader, device, max_batches=2))

## 5. Checkpoint và resume state

Checkpoint không chỉ có weights: optimizer, step, history, tokenizer, RNG và data manifest đều cần để tái lập run.

In [ ]:
notebook_checkpoint = Path(r'C:/Users/Admin/Documents/Codex/2026-07-21/https-github-com-rasbt-llms-from/notebook-step-by-step.pt')
save_checkpoint(notebook_checkpoint, model, small_train_cfg, tokenizer.to_state(), history, optimizer, step=5, data_manifest=manifest.to_dict())
loaded = load_checkpoint(notebook_checkpoint, device)
print('checkpoint step:', loaded['step'])
print('stored manifest hash:', loaded['data_manifest']['document_sha256'])
print('stored optimizer tensors:', len(loaded['optimizer_state']['state']))

## 6. KV cache: logits parity và generation parity

Full-prefix logits và chunked cached logits phải giống nhau trong `eval()` mode.

In [ ]:
model.eval()
probe = torch.randint(0, cfg.vocab_size, (1, 10), device=device)
full_logits, _ = model(probe)
_, past = model(probe[:, :6], use_cache=True)
cached_logits, past = model(probe[:, 6:], past_key_values=past, use_cache=True)
print('cached logits max error:', (full_logits[:, 6:] - cached_logits).abs().max().item())
uncached_ids = model.generate_uncached(probe[:, :8], max_new_tokens=12)
cached_ids = model.generate_cached(probe[:, :8], max_new_tokens=12)
print('generation equal:', torch.equal(uncached_ids, cached_ids), 'generated shape:', tuple(cached_ids.shape))

## 7. Benchmark interpretation

Benchmark production dùng checkpoint và nhiều trial; CPU model rất nhỏ có thể không nhanh hơn vì Python overhead. Trên GPU/context dài, hãy tập trung vào `cached_incremental_decode` và `retained_kv_cache_bytes`.

In [ ]:
from llm_lab.benchmark import benchmark_scenario
benchmark_prompt = torch.randint(0, cfg.vocab_size, (1, 8), device=device)
benchmark_result = benchmark_scenario(model, benchmark_prompt, steps=4, trials=2, warmup=1, device=device)
print(json.dumps(benchmark_result, indent=2))

## Kết luận

Sau notebook này, hãy tăng `context_length` lên 128/256, chạy `configs/tinystories_tiny.json` trên Colab, rồi mới thay `MultiHeadAttention` bằng GQA. Giữ nguyên data split, tokenizer, seed, checkpoint budget và benchmark.